# ricotta · interpretability for LM post-training, end to end

A guided tour of every layer of [ricotta](https://github.com/JacopoCirica/ricotta) on **Qwen3.5-4B** (a hybrid linear-attention model) and its reasoning post-train, unified by one question: **what did post-training change?**

- **attn** — where the model looks (incl. *exact effective-attention* for the Gated DeltaNet layers that have no softmax matrix)
- **attrib** — which inputs drive / changed the output; CoT faithfulness; modality
- **probe / hidden** — when the answer becomes decodable; per-layer logit lens; representation drift (CKA)
- **tts** — True Thinking Score: which reasoning steps are causally load-bearing

> Runtime → Change runtime type → **A100 GPU**. The `circuits` layer is skipped here — it needs transcoders + an older transformers, and can't analyze Qwen3.5's linear-attention layers (see the repo for a Qwen3 circuits notebook).

In [ ]:
%pip install -q "git+https://github.com/JacopoCirica/ricotta.git"
%pip install -q -U "transformers>=4.58" scikit-learn   # Qwen3.5 needs recent transformers
import torch; print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
import torch

BASE = "Qwen/Qwen3.5-4B-Base"   # pre-trained
FT   = "Qwen/Qwen3.5-4B"        # released reasoning post-train (swap in your own SDPO checkpoint here)
DTYPE = torch.bfloat16

## 1 · attn — exact effective-attention for hybrid linear attention

Qwen3.5 interleaves 8 softmax layers with 24 **Gated DeltaNet** layers that have no attention matrix. ricotta reconstructs their *exact* effective attention (`o_t = Σ A[t,j] v_j`) by re-running the layer's own recurrence with indicator values — verified by reconstruction to ~1e-7.

In [ ]:
from ricotta import get_hybrid_attention
from ricotta.attn import hybrid_heatmap

data = get_hybrid_attention(FT, "The capital of the state containing Dallas is Austin.")
print(data)   # 32 layers [8 full + 24 linear], max_recon_err ~1e-7
print('full-attention layers at original indices:', [l.layer for l in data.full()])

In [ ]:
# a reconstructed linear-attention layer: note SIGNED weights (blue cells) — the
# delta rule subtracts; rows are NOT a probability distribution (unlike softmax)
hybrid_heatmap(data, layer=30)

In [ ]:
# a real softmax layer of the same model, for contrast (attention-sink, all positive)
hybrid_heatmap(data, layer=3)

## 2 · attrib — which input tokens drive a prediction

Gradient / integrated-gradients / occlusion attribution. Works on Qwen3.5 because it's purely gradient/ablation-based — no attention matrix or transcoders needed.

In [ ]:
from ricotta import LM, occlusion, show_attribution

lm_ft = LM.load(FT, dtype=DTYPE)
ids = lm_ft.encode("The capital of the state containing Dallas is Austin.")
austin = lm_ft.token_strings(ids).index(" Austin")
attr = occlusion(lm_ft, ids, target_pos=austin, source_window=8)
print('drivers of predicting " Austin":', [(t, round(v,2)) for _,t,v in attr.top(5)])
show_attribution(attr)

## 3 · attrib — checkpoint diff (what did the reasoning post-train change?)

Per-token log-prob difference between base and fine-tune on the same text: green = the post-train assigns more probability, red = the base does.

In [ ]:
from ricotta import token_logprob_diff, show_diff

lm_base = LM.load(BASE, dtype=DTYPE)
ids = lm_base.encode("Let's solve this step by step. 17 times 23 equals 391.")
diff = token_logprob_diff(lm_base, lm_ft, ids)
print('largest shifts:', diff.top(6))
show_diff(diff)

## 4 · attrib — chain-of-thought faithfulness

Generate a reasoning chain with the post-train, then ablate each step and measure the drop in the answer's probability. A large drop = the step is *load-bearing* (the answer genuinely depends on it).

In [ ]:
from ricotta import Span, cot_faithfulness

# build a short reasoning trace from labeled chunks so step spans are exact
chunks = [
    ("prompt", "Q: A shop has 3 boxes of 8 pens, then sells 5 pens. How many pens remain? Reason step by step.\n"),
    ("step1", "There are 3 boxes of 8 pens, so 3 x 8 = 24 pens."),
    ("step2", " The shop sells 5 pens, so 24 - 5 = 19."),
    ("answer", " The answer is 19"),
]
parts, spans, pos = [], {}, 0
for name, text in chunks:
    p = lm_ft.tokenizer(text, add_special_tokens=False, return_tensors='pt').input_ids
    parts.append(p); spans[name] = (pos, pos + p.shape[1]); pos += p.shape[1]
ids = torch.cat(parts, 1).to(lm_ft.device)
answer_pos = spans['answer'][1] - 1
step_spans = [Span(*spans[s], label=s) for s in ('step1','step2')]

res = cot_faithfulness(lm_ft, ids, answer_pos=answer_pos, step_spans=step_spans)
for s in res['ranked']:
    print(f"{s.span.label}: answer-prob drop {s.ablation_drop:+.4f}")

## 5 · probe — when does the answer become decodable? (belief probing)

Reproduces *Exploring belief states in LLM chains of thought*: train a linear probe on residual-stream activations per (layer, position) to decode the model's own answer. A100 makes the generation fast, so we can use more questions.

In [ ]:
import random
from ricotta import Question, run_belief_experiment
from ricotta.probe import belief_curve

random.seed(0)
questions = []
for _ in range(80):
    a, b = random.randint(100, 9999), random.randint(100, 9999)
    questions.append(Question(f"Is {a} greater than {b}?", "Yes", "No"))

res = run_belief_experiment(lm_ft, questions, n_positions=40, max_new_tokens=160)
print(res, '| peak:', res.peak())
belief_curve(res)

## 6 · hidden — per-layer logit lens

Unembed each layer's residual stream (after the final norm) to read the model's running next-token guess across depth — no training.

In [ ]:
from ricotta import logit_lens

ll = logit_lens(lm_ft, "The capital of France is")
print('top-1 next-token guess across layers:', ll.trajectory(pos=-1))

## 7 · hidden — representation drift across checkpoints (CKA)

Linear CKA between base and post-train hidden states on the same prompts, per layer. Low CKA = the layer post-training reshaped the most — the diff-two-checkpoints verb applied to representations.

In [ ]:
from ricotta import cka_drift

prompts = [
    "The mitochondria is the powerhouse of the cell.",
    "To solve for x, subtract 3 from both sides of the equation.",
    "Once upon a time, in a village by the sea, there lived a fisherman.",
]
drift = cka_drift(lm_base, lm_ft, prompts, last_n=16)
print('layers most reshaped by post-training (lowest CKA):', drift.most_changed(5))

## 8 · the unified report

One call runs the attribution diff over a probe set and emits a single markdown report — drop it into an eval loop to track each checkpoint.

In [ ]:
from ricotta import posttrain_report

md = posttrain_report(BASE, FT, prompts=[
    "The answer to 6 times 7 is 42.",
    "Water boils at 100 degrees Celsius at sea level.",
], attribution_method="occlusion")
print(md)

## 9 · tts — True Thinking Score (causal contribution of each reasoning step)

The rigorous version of CoT faithfulness ([arXiv:2510.24941](https://arxiv.org/abs/2510.24941) / Thinking Machines recipe): generate a reasoning chain, perturb the numbers in each step (offsets in {-3..3}), and measure how `P(correct answer)` changes under a 2x2 of {context, step} intact/perturbed. necessity + sufficiency -> TTS. Steps >= 0.7 are **true-thinking** (causally load-bearing), <= 0.005 are **decorative**.

> Speed tip: on the A100, `!pip install -q flash-linear-attention causal-conv1d` swaps Qwen3.5's slow torch-fallback linear attention for the fast kernel.

In [ ]:
from ricotta import true_thinking_score, tts_chart
from ricotta.tts import generate_reasoning

q   = ("A store has 33 red, 19 blue, and 14 green marbles. It sells 6 red, 4 blue, "
       "and 2 green. How many marbles are left?")
ans = "54"   # correct answer (scored inside \\boxed{})

prompt, reasoning = generate_reasoning(lm_ft, q, max_new_tokens=1024)
res = true_thinking_score(lm_ft, prompt, reasoning, ans, verbose=True)
for s in res.steps:
    sv = " [self-verify]" if s.self_verification else ""
    print(f"step {s.index}: TTS={s.tts:.3f} ({s.label}){sv}  {s.text[:60]!r}")
tts_chart(res)

---
That's every Qwen3.5-compatible layer of ricotta: attention (incl. the novel effective-attention reconstruction), input attribution, CoT faithfulness, belief probing, logit lens, CKA drift, and the unified report. Swap `FT` for your own fine-tune to analyze what *your* post-training changed.